In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\sansu\Downloads\education_dataset (1).csv")
df.head()
from sklearn.feature_selection import VarianceThreshold



In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              2000 non-null   int64  
 1   age                     2000 non-null   int64  
 2   study_hours_per_week    2000 non-null   float64
 3   attendance_rate         2000 non-null   float64
 4   homework_completion     2000 non-null   float64
 5   reading_score           2000 non-null   float64
 6   math_score              2000 non-null   float64
 7   science_score           2000 non-null   float64
 8   parent_education_level  2000 non-null   object 
 9   family_income           2000 non-null   float64
 10  student_ethnicity       2000 non-null   object 
 11  disability_status       2000 non-null   object 
 12  tutoring_support        2000 non-null   object 
 13  internet_access         2000 non-null   object 
 14  school_type             2000 non-null   

In [3]:
# Diagnostic cell - run inside the same notebook kernel
import sys, traceback

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("sys.path[0:6]:", sys.path[:6])

try:
    import sklearn
    print("scikit-learn import OK, version:", sklearn.__version__)
except Exception:
    print("scikit-learn import failed. Traceback:")
    traceback.print_exc()

import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install scikit-learn

Python executable: c:\Users\sansu\AppData\Local\Programs\Python\Python313\python.exe
Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
sys.path[0:6]: ['c:\\Users\\sansu\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'c:\\Users\\sansu\\AppData\\Local\\Programs\\Python\\Python313\\DLLs', 'c:\\Users\\sansu\\AppData\\Local\\Programs\\Python\\Python313\\Lib', 'c:\\Users\\sansu\\AppData\\Local\\Programs\\Python\\Python313', '', 'C:\\Users\\sansu\\AppData\\Roaming\\Python\\Python313\\site-packages']
scikit-learn import OK, version: 1.7.2


In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# 1. Select numeric columns (avoid the previous include=[''] bug)
numeric = df.select_dtypes(include=[np.number]).copy()
print("Numeric feature count:", numeric.shape[1])

# 2. Handle missing values: fill with column mean (or drop columns with too many NaNs)
numeric = numeric.fillna(numeric.mean())

# 3. Inspect variances to choose a threshold
variances = numeric.var(axis=0)
print("Smallest variances (ascending):")
print(variances.sort_values().head(20))

# 4. Choose threshold
# - threshold=0 removes constant columns (variance == 0)
# - choose a small positive value (e.g. 0.01 or 0.1) depending on your scale and domain
threshold = 0.0

selector = VarianceThreshold(threshold=threshold)
selector.fit(numeric)

# 5. Get selected feature names and reduced DataFrame
mask = selector.get_support()
selected_cols = numeric.columns[mask]
removed_cols = numeric.columns[~mask]

print(f"Kept {len(selected_cols)} / {numeric.shape[1]} numeric features")
print("Removed columns:", list(removed_cols))

numeric_reduced = numeric.loc[:, selected_cols]

# 6. If you want the full DataFrame with non-numeric preserved:
non_numeric = df.select_dtypes(exclude=[np.number]).reset_index(drop=True)
numeric_reduced = numeric_reduced.reset_index(drop=True)
df_reduced = pd.concat([non_numeric, numeric_reduced], axis=1)

Numeric feature count: 14
Smallest variances (ascending):
bus_arrival_time        6.701101e-01
num_siblings            1.228550e+00
gaming_hours            1.443016e+00
age                     2.047451e+00
social_media_hours      2.271643e+00
study_hours_per_week    8.853571e+00
final_exam_score        5.426186e+01
reading_score           1.105554e+02
science_score           1.184065e+02
attendance_rate         1.321353e+02
math_score              1.340775e+02
homework_completion     2.066884e+02
student_id              3.335000e+05
family_income           2.318106e+08
dtype: float64
Kept 14 / 14 numeric features
Removed columns: []


In [4]:
# Step A: encode categorical columns so the model can use them
# Run this after any previous cleaning (e.g., df_reduced or df)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("Categorical columns detected:", cat_cols)

# If there are no categorical columns this will do nothing
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# quick check
print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)

Categorical columns detected: ['parent_education_level', 'student_ethnicity', 'disability_status', 'tutoring_support', 'internet_access', 'school_type', 'region', 'locker_number', 'favorite_subject', 'passed_course', 'college_admission']
Shape before encoding: (2000, 25)
Shape after encoding: (2000, 836)


In [7]:
# Step B: split, scale, fit LinearRegression, evaluate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# === SET TARGET HERE ===
y_col = 'YOUR_TARGET_COLUMN'   # <-- replace with the column you want to predict

# Basic checks
if y_col not in df_encoded.columns:
    raise KeyError(f"Target column '{y_col}' not found. Available columns: {list(df_encoded.columns[:20])} ...")

# Features and target
X = df_encoded.drop(columns=[y_col])
y = df_encoded[y_col]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (recommended for many models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit a linear regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predict & evaluate
y_pred = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"R^2 on test set: {r2:.4f}")
print(f"RMSE on test set: {rmse:.4f}")

# Optional: show top coefficients (absolute) to understand important features
import numpy as np
coef = model.coef_
feature_names = X.columns
top_idx = np.argsort(np.abs(coef))[::-1][:10]
print("\nTop features by absolute coefficient:")
for i in top_idx:
    print(f"  {feature_names[i]}: coef={coef[i]:.4f}")

KeyError: "Target column 'YOUR_TARGET_COLUMN' not found. Available columns: ['student_id', 'age', 'study_hours_per_week', 'attendance_rate', 'homework_completion', 'reading_score', 'math_score', 'science_score', 'family_income', 'social_media_hours', 'gaming_hours', 'num_siblings', 'bus_arrival_time', 'final_exam_score', 'parent_education_level_graduate', 'parent_education_level_high school', 'parent_education_level_none', 'student_ethnicity_black', 'student_ethnicity_hispanic', 'student_ethnicity_other'] ..."